### Блокнот чисто для тестов со Spark. **Dev-only**
Происходит преобразование сырых данных с ingestion layer в очищенные дедуплицированные строки с преобразованием типов.
TODO: надо сделать партицирование по айди монеты

In [ ]:
try: spark.stop()
except: pass

from pyspark.sql import SparkSession

PACKAGES = ",".join([
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.5",
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "com.amazonaws:aws-java-sdk-bundle:1.12.262",
])

spark = (SparkSession.builder
    .appName("dev")
    .master("local[*]")
    .config("spark.jars.packages", PACKAGES)
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
df = spark.read.parquet("s3a://crypto-lake/bronze/crypto.prices/")
bronze_schema = df.schema  # сериализованная схема
print(bronze_schema)
df.select("raw_json").show(1, truncate=200)

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, FloatType, IntegerType

TOPIC = "crypto.prices"
CHECKPOINT_BRONZE_PATH = f"s3a://spark-checkpoints/bronze/{TOPIC}"
CHECKPOINT_SILVER_PATH = f"s3a://spark-checkpoints/silver/{TOPIC}"
BRONZE_PATH = f"s3a://crypto-lake/bronze/{TOPIC}"
SILVER_PATH = f"s3a://crypto-lake/silver/{TOPIC}"

# схема вложенной структуры полезной нагрузки
coin_schema = StructType([
    StructField("id", StringType(), True),
    StructField("symbol", StringType(), True),
    StructField("name", StringType(), True),
    StructField("image", StringType(), True),
    StructField("current_price", StringType(), True),
    StructField("market_cap", StringType(), True),
    StructField("market_cap_rank", StringType(), True),
    StructField("fully_diluted_valuation", StringType(), True),
    StructField("total_volume", StringType(), True),
    StructField("high_24h", StringType(), True),
    StructField("low_24h", StringType(), True),
    StructField("price_change_24h", StringType(), True),
    StructField("price_change_percentage_24h", StringType(), True),
    StructField("market_cap_change_24h", StringType(), True),
    StructField("market_cap_change_percentage_24h", StringType(), True),
    StructField("circulating_supply", StringType(), True),
    StructField("total_supply", StringType(), True),
    StructField("max_supply", StringType(), True),
    StructField("ath", StringType(), True),
    StructField("ath_change_percentage", StringType(), True),
    StructField("ath_date", StringType(), True),
    StructField("atl", StringType(), True),
    StructField("atl_change_percentage", StringType(), True),
    StructField("atl_date", StringType(), True),
    StructField("last_updated", StringType(), True),
])

payload_schema = StructType([
    StructField("ts", LongType(), True),
    StructField("type", StringType(), True),
    StructField("cs", LongType(), True),
    StructField("topic", StringType(), True),
    StructField("data", data_schema, True),
])

PRICE_T  = DecimalType(28, 8)
CAP_T    = DecimalType(28, 2)
VOL_T    = DecimalType(28, 2)
SUPPLY_T = DecimalType(28, 8)
PCT_T    = FloatType()
RANK_T   = IntegerType()

source = (spark.readStream
    .format("parquet")
    .schema(bronze_schema) 
    .option("maxFilesPerTrigger", 100)
    .load(BRONZE_PATH))

clean = (source
    .withColumn("p", F.from_json(F.col("raw_json"), payload_schema))
    .select(
        F.col("ingestion_ts"),
        F.col("kafka_ts"),
        F.to_timestamp("p.last_updated").alias("event_time"),
        F.col("p.id").alias("coin_id"),
        F.upper(F.col("p.symbol")).alias("symbol"),
        F.col("p.name").alias("name"),
        F.col("p.image").alias("image_url"),
        F.col("p.current_price").cast(PRICE_T).alias("current_price"),
        F.col("p.market_cap").cast(CAP_T).alias("market_cap"),
        F.col("p.market_cap_rank").cast(RANK_T).alias("market_cap_rank"),
        F.col("p.fully_diluted_valuation").cast(CAP_T).alias("fully_diluted_valuation"),
        F.col("p.total_volume").cast(VOL_T).alias("total_volume"),
        F.col("p.high_24h").cast(PRICE_T).alias("high_24h"),
        F.col("p.low_24h").cast(PRICE_T).alias("low_24h"),
        F.col("p.price_change_24h").cast(PRICE_T).alias("price_change_24h"),
        F.col("p.price_change_percentage_24h").cast(PCT_T).alias("price_change_pct_24h"),
        F.col("p.market_cap_change_24h").cast(CAP_T).alias("market_cap_change_24h"),
        F.col("p.market_cap_change_percentage_24h").cast(PCT_T).alias("market_cap_change_pct_24h"),
        F.col("p.circulating_supply").cast(SUPPLY_T).alias("circulating_supply"),
        F.col("p.total_supply").cast(SUPPLY_T).alias("total_supply"),
        F.col("p.max_supply").cast(SUPPLY_T).alias("max_supply"),
        F.col("p.ath").cast(PRICE_T).alias("ath"),
        F.col("p.ath_change_percentage").cast(PCT_T).alias("ath_change_pct"),
        F.to_timestamp("p.ath_date").alias("ath_date"),
        F.col("p.atl").cast(PRICE_T).alias("atl"),
        F.col("p.atl_change_percentage").cast(PCT_T).alias("atl_change_pct"),
        F.to_timestamp("p.atl_date").alias("atl_date"),
    )
    .dropDuplicates(["coin_id", "event_time"])
    .where(F.col("current_price").isNotNull() & (F.col("current_price") > 0))
    .withColumn("year",  F.year("event_time"))
    .withColumn("month", F.month("event_time"))
    .withColumn("day",   F.dayofmonth("event_time"))
)

write = (clean.writeStream
    .format("console")
    .option("truncate", False)
    .trigger(processingTime="60 seconds")
    .start())
write.awaitTermination()

'''write = (clean.writeStream
    .format("parquet")
    .option("path", SILVER_PATH)
    .option("checkpointLocation", CHECKPOINT_SILVER_PATH)
    .partitionBy("id", "year", "month", "day")
    .trigger(processingTime="60 seconds")
    .start())'''

write.awaitTermination()

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, ArrayType,
    DecimalType, FloatType, IntegerType
)

PRICE_T, CAP_T, VOL_T, SUPPLY_T = (
    DecimalType(28, 8), DecimalType(28, 2),
    DecimalType(28, 2), DecimalType(28, 8),
)
PCT_T, RANK_T = FloatType(), IntegerType()

coin_schema = StructType([
    StructField("id", StringType()),
    StructField("symbol", StringType()),
    StructField("name", StringType()),
    StructField("image", StringType()),
    StructField("current_price", StringType()),
    StructField("market_cap", StringType()),
    StructField("market_cap_rank", StringType()),
    StructField("fully_diluted_valuation", StringType()),
    StructField("total_volume", StringType()),
    StructField("high_24h", StringType()),
    StructField("low_24h", StringType()),
    StructField("price_change_24h", StringType()),
    StructField("price_change_percentage_24h", StringType()),
    StructField("market_cap_change_24h", StringType()),
    StructField("market_cap_change_percentage_24h", StringType()),
    StructField("circulating_supply", StringType()),
    StructField("total_supply", StringType()),
    StructField("max_supply", StringType()),
    StructField("ath", StringType()),
    StructField("ath_change_percentage", StringType()),
    StructField("ath_date", StringType()),
    StructField("atl", StringType()),
    StructField("atl_change_percentage", StringType()),
    StructField("atl_date", StringType()),
    StructField("last_updated", StringType()),
])
payload_schema = ArrayType(coin_schema)

spark = (SparkSession.builder
    .appName("silver_prices")
    .getOrCreate())

bronze = (spark.readStream
    .schema(spark.read.parquet("s3a://crypto-lake/bronze/crypto.prices/").schema)
    .parquet("s3a://crypto-lake/bronze/crypto.prices/"))

clean = (bronze
    .withColumn("arr", F.from_json("raw_json", payload_schema))
    .withColumn("p", F.explode("arr"))
    .select(
        F.col("ingestion_ts"),
        F.col("kafka_ts"),
        F.to_timestamp("p.last_updated").alias("event_time"),
        F.col("p.id").alias("coin_id"),
        F.upper("p.symbol").alias("symbol"),
        F.col("p.name").alias("name"),
        F.col("p.image").alias("image_url"),
        F.col("p.current_price").cast(PRICE_T).alias("current_price"),
        F.col("p.market_cap").cast(CAP_T).alias("market_cap"),
        F.col("p.market_cap_rank").cast(RANK_T).alias("market_cap_rank"),
        F.col("p.fully_diluted_valuation").cast(CAP_T).alias("fully_diluted_valuation"),
        F.col("p.total_volume").cast(VOL_T).alias("total_volume"),
        F.col("p.high_24h").cast(PRICE_T).alias("high_24h"),
        F.col("p.low_24h").cast(PRICE_T).alias("low_24h"),
        F.col("p.price_change_24h").cast(PRICE_T).alias("price_change_24h"),
        F.col("p.price_change_percentage_24h").cast(PCT_T).alias("price_change_pct_24h"),
        F.col("p.market_cap_change_24h").cast(CAP_T).alias("market_cap_change_24h"),
        F.col("p.market_cap_change_percentage_24h").cast(PCT_T).alias("market_cap_change_pct_24h"),
        F.col("p.circulating_supply").cast(SUPPLY_T).alias("circulating_supply"),
        F.col("p.total_supply").cast(SUPPLY_T).alias("total_supply"),
        F.col("p.max_supply").cast(SUPPLY_T).alias("max_supply"),
        F.col("p.ath").cast(PRICE_T).alias("ath"),
        F.col("p.ath_change_percentage").cast(PCT_T).alias("ath_change_pct"),
        F.to_timestamp("p.ath_date").alias("ath_date"),
        F.col("p.atl").cast(PRICE_T).alias("atl"),
        F.col("p.atl_change_percentage").cast(PCT_T).alias("atl_change_pct"),
        F.to_timestamp("p.atl_date").alias("atl_date"),
    )
    .where(F.col("current_price").isNotNull() & (F.col("current_price") > 0))
    .withWatermark("event_time", "10 minutes")
    .dropDuplicates(["coin_id", "event_time"])
    .withColumn("year",  F.year("event_time"))
    .withColumn("month", F.month("event_time"))
    .withColumn("day",   F.dayofmonth("event_time"))
)
'''
(clean.writeStream
    .format("parquet")
    .option("path", "s3a://crypto-lake/silver/prices_clean/")
    .option("checkpointLocation", "s3a://crypto-lake/silver/_checkpoints/prices_clean/")
    .partitionBy("year", "month", "day")
    .outputMode("append")
    .trigger(processingTime="60 seconds")
    .start()
    .awaitTermination())'''

query = (clean.writeStream
    .format("console")
    .option("truncate", False)
    .option("numRows", 10)
    .trigger(processingTime="60 seconds")
    .start())
query.awaitTermination()